# 02 — Skeleton Extraction

Batch-process frames through YOLOv8-Pose to extract dual-player skeletons.

**Steps:**
1. Process FineBadminton frames → per-rally skeleton .npy files
2. Process ShuttleSet extracted frames (25 matches) → per-shot skeleton .npy files
3. Compute enriched node features (L0–L3)
4. Quality validation and visualization

In [3]:
!pip install tqdm

In [4]:
import sys
sys.path.insert(0, '..')

import json
import numpy as np
import cv2
from pathlib import Path
from collections import defaultdict
from tqdm import tqdm
import matplotlib.pyplot as plt

from src.config import *
from src.data.pose_extractor import PoseExtractor
from src.data.graph_builder import GraphBuilder
from src.data.feature_eng import FeatureEngineer

## 1. Extract FineBadminton Skeletons

Frames are pre-provided as `{match}_{rally}_{frame}.jpg`.
Group by rally, extract skeletons, save one `.npy` per rally.

In [5]:
extractor = PoseExtractor()
FB_SKELETONS.mkdir(parents=True, exist_ok=True)

frame_files = sorted(FB_FRAMES.glob('*.jpg'))
print(f"Total FineBadminton frames: {len(frame_files)}")

# Group by rally (first two parts of filename)
rally_frames = defaultdict(list)
for f in frame_files:
    parts = f.stem.split('_')
    if len(parts) >= 3:
        rally_id = f"{parts[0]}_{parts[1]}"  # e.g., "0011_002"
        frame_num = int(parts[2])
        rally_frames[rally_id].append((frame_num, f))

print(f"Rallies found: {len(rally_frames)}")
for rally_id in sorted(rally_frames.keys())[:10]:
    print(f"  {rally_id}: {len(rally_frames[rally_id])} frames")

Total FineBadminton frames: 10620
Rallies found: 40
  0011_001: 327 frames
  0011_002: 643 frames
  0011_003: 238 frames
  0011_004: 191 frames
  0011_005: 639 frames
  0012_001: 264 frames
  0012_002: 652 frames
  0012_003: 225 frames
  0013_001: 118 frames
  0013_002: 356 frames


In [ ]:
# Process each rally (skip if already done)
for rally_id, frame_list in tqdm(sorted(rally_frames.items()), desc='FB Rallies'):
    output_path = FB_SKELETONS / f"{rally_id}.npy"
    if output_path.exists():
        continue
    
    frame_list.sort(key=lambda x: x[0])
    frames = [cv2.imread(str(f)) for _, f in frame_list]
    
    skeleton = extractor.extract_sequence(frames)
    np.save(output_path, skeleton)

fb_skel_files = sorted(FB_SKELETONS.glob('*.npy'))
print(f"\nSaved skeletons to {FB_SKELETONS}")
print(f"Skeleton files: {len(fb_skel_files)}")

FB Rallies:   0%|          | 0/40 [00:00<?, ?it/s]

[INFO] Loaded yolov8x-pose


FB Rallies:  72%|███████▎  | 29/40 [1:58:45<1:05:03, 354.83s/it]

## 2. Extract ShuttleSet Skeletons

25 matches already have frames extracted in `datasets_preprocessing/ShuttleSet/frames/`.
Each match has `frame_XXXXXX.png` files. For each shot in the JSON outputs,
we extract the 3-frame window (prev, curr, next) skeletons.

In [ ]:
SS_SKELETONS.mkdir(parents=True, exist_ok=True)

# Load all shot records from JSON outputs
match_dirs = sorted([d for d in SS_FRAMES.iterdir() if d.is_dir()])
print(f"Match frame directories available: {len(match_dirs)}")

# Show frame counts per match
for md in match_dirs[:5]:
    fc = len(list(md.glob('*.png')))
    print(f"  {md.name}: {fc} frames")

In [ ]:
# For each match: extract skeletons from frames
# Strategy: process all frames in a match sequentially,
# save per-match skeleton arrays

for match_dir in tqdm(match_dirs, desc='SS Matches'):
    match_id = match_dir.name
    output_path = SS_SKELETONS / f"{match_id}.npy"
    
    if output_path.exists():
        continue
    
    frame_files = sorted(match_dir.glob('*.png'))
    if not frame_files:
        print(f"  [SKIP] No frames: {match_id}")
        continue
    
    print(f"  Processing {match_id}: {len(frame_files)} frames...")
    
    # Process in batches to avoid loading all frames at once
    all_keypoints = []
    for frame_path in tqdm(frame_files, desc=f'  {match_id}', leave=False):
        frame = cv2.imread(str(frame_path))
        if frame is None:
            all_keypoints.append(None)
            continue
        
        kpts = extractor.extract_frame(frame)
        all_keypoints.append(kpts)  # (2, 17, 2) or None
    
    # Build skeleton array: (2, T, 34)
    T = len(all_keypoints)
    skeleton = np.zeros((T, 2, NUM_JOINTS, 2), dtype=np.float32)
    
    for t, kpts in enumerate(all_keypoints):
        if kpts is not None:
            skeleton[t] = kpts
        elif t > 0:
            skeleton[t] = skeleton[t-1]  # forward-fill
    
    # Reshape: (T, 2_players, 17_joints, 2_xy) → (2_xy, T, 34_nodes)
    skeleton = skeleton.reshape(T, -1, 2)  # (T, 34, 2)
    skeleton = skeleton.transpose(2, 0, 1)  # (2, T, 34)
    
    np.save(output_path, skeleton)
    print(f"  Saved: {skeleton.shape}")

ss_skel_files = sorted(SS_SKELETONS.glob('*.npy'))
print(f"\nTotal ShuttleSet skeleton files: {len(ss_skel_files)}")

## 3. Test Feature Engineering Layers

In [ ]:
# Load a sample skeleton and test all feature layers
sample_files = sorted(FB_SKELETONS.glob('*.npy'))

if sample_files:
    skel = np.load(sample_files[0])  # (2, T, 34)
    print(f"Raw skeleton: {skel.shape}")
    
    for layer in ['L0', 'L1', 'L2', 'L3']:
        eng = FeatureEngineer(feature_layer=layer)
        features = eng.compute(skel)
        print(f"  {layer}: {features.shape} — {FEATURE_DIMS[layer]} features/node")
else:
    print("No skeleton files yet. Run extraction cells above.")
    # Test with synthetic data
    skel = np.random.randn(2, 16, 34).astype(np.float32) * 100
    for layer in ['L0', 'L1', 'L2', 'L3']:
        eng = FeatureEngineer(feature_layer=layer)
        features = eng.compute(skel)
        print(f"  {layer}: {features.shape} (synthetic data)")

## 4. Validate Skeleton Quality

In [ ]:
# Visualize skeleton trajectories for a FineBadminton rally
if sample_files:
    skel = np.load(sample_files[0])
    print(f"Skeleton: {skel.shape}  (rally: {sample_files[0].stem})")
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    axes[0, 0].plot(skel[0, :, :17], alpha=0.4)
    axes[0, 0].set_title('Player 1 — X over time')
    axes[0, 0].set_xlabel('Frame')
    
    axes[0, 1].plot(skel[1, :, :17], alpha=0.4)
    axes[0, 1].set_title('Player 1 — Y over time')
    
    axes[1, 0].plot(skel[0, :, 17:], alpha=0.4)
    axes[1, 0].set_title('Player 2 — X over time')
    
    axes[1, 1].plot(skel[1, :, 17:], alpha=0.4)
    axes[1, 1].set_title('Player 2 — Y over time')
    
    plt.suptitle(f'Skeleton Trajectories: {sample_files[0].stem}')
    plt.tight_layout()
    plt.show()

In [ ]:
# Visualize L1 (velocity) features for a sample
if sample_files:
    eng = FeatureEngineer(feature_layer='L1')
    features = eng.compute(skel)
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # Velocity magnitude per joint over time
    vx = features[2]  # (T, V)
    vy = features[3]
    vel_mag = np.sqrt(vx**2 + vy**2)
    
    # Player 1 wrist velocity (joint 9 = left wrist, 10 = right wrist)
    axes[0].plot(vel_mag[:, 9], label='Left wrist', linewidth=2)
    axes[0].plot(vel_mag[:, 10], label='Right wrist', linewidth=2)
    axes[0].set_title('Player 1 — Wrist Velocity Magnitude')
    axes[0].set_xlabel('Frame')
    axes[0].legend()
    
    # Player 2 wrist velocity
    axes[1].plot(vel_mag[:, 9+17], label='Left wrist', linewidth=2)
    axes[1].plot(vel_mag[:, 10+17], label='Right wrist', linewidth=2)
    axes[1].set_title('Player 2 — Wrist Velocity Magnitude')
    axes[1].set_xlabel('Frame')
    axes[1].legend()
    
    # Velocity heatmap (all joints)
    im = axes[2].imshow(vel_mag.T, aspect='auto', cmap='hot')
    axes[2].set_xlabel('Frame')
    axes[2].set_ylabel('Joint index')
    axes[2].set_title('Velocity Magnitude Heatmap (all joints)')
    axes[2].axhline(y=16.5, color='cyan', linestyle='--', alpha=0.5)
    plt.colorbar(im, ax=axes[2])
    
    plt.tight_layout()
    plt.show()

## 5. Test Dataset Loading

In [ ]:
from src.data.dataset import FineBadmintonDataset, ShuttleSetDataset

# Test FineBadminton dataset
fb_ds = FineBadmintonDataset(feature_layer='L2')
print(f"\nFineBadminton samples: {len(fb_ds)}")

if len(fb_ds) > 0:
    x, y = fb_ds[0]
    print(f"  Sample shape: {x.shape}, label: {y} ({IDX_TO_STRATEGY[y]})")
    print(f"  Has real data: {x.abs().sum() > 0}")

# Test ShuttleSet dataset
ss_ds = ShuttleSetDataset(feature_layer='L2')
print(f"\nShuttleSet samples: {len(ss_ds)}")

if len(ss_ds) > 0:
    sample = ss_ds[0]
    if isinstance(sample, tuple):
        x, st = sample
        print(f"  Sample shape: {x.shape}, shot_type: {st}")
    else:
        print(f"  Sample shape: {sample.shape}")

In [ ]:
# Test fold splits
splits = fb_ds.get_fold_splits(n_folds=5)
print(f"Folds: {len(splits)}")
for i, (train, val, test) in enumerate(splits):
    print(f"  Fold {i+1}: train={len(train)}, val={len(val)}, test={len(test)}")